# WM-811K Wafer Defect Classification — CNN Pipeline

**Goal:** Train a CNN to classify wafer maps into 9 classes (8 defect types + "none").  
**Principles:** No data leakage · Stratified splits · Augmentation on train only · Early stopping · Class-weighted loss.

## 1. Imports & Configuration

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.preprocessing import LabelEncoder

@dataclass
class CFG:
    seed: int = 42
    batch_size: int = 64
    lr: float = 1e-3
    max_epochs: int = 100
    es_patience: int = 10      # early stopping patience
    lr_patience: int = 5       # ReduceLROnPlateau patience
    lr_factor: float = 0.5
    val_frac: float = 0.20
    num_workers: int = 2
    checkpoint: str = '../best_model.pt'
    figures_dir: str = '../figures'

cfg = CFG()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(cfg.seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
Path(cfg.figures_dir).mkdir(parents=True, exist_ok=True)

## 2. Data Loading & Preparation

We reuse the unpacking pattern from the EDA notebook to unpack the nested arrays in `lotName`, `trainTestLabel`, and `failureType`.

In [ ]:
df = pd.read_pickle('../data/LSWMD_modern.pkl')

# Fix typo from original dataset
df.rename(columns={'trianTestLabel': 'trainTestLabel'}, inplace=True)

def unpack(x):
    arr = np.asarray(x)
    if arr.size == 0:
        return None
    return arr[0][0]

for col in ['lotName', 'trainTestLabel', 'failureType']:
    df[col] = df[col].apply(unpack)

# Keep only labeled rows with a known failure type
labeled = df[df['trainTestLabel'].notna() & df['failureType'].notna()].copy()
print(f'Labeled rows: {len(labeled):,}')
print(labeled['failureType'].value_counts())

In [ ]:
# Encode failure type to integer labels — deterministic alphabetical order
le = LabelEncoder()
labeled['label'] = le.fit_transform(labeled['failureType'])
CLASS_NAMES = list(le.classes_)
N_CLASSES = len(CLASS_NAMES)
print(f'Classes ({N_CLASSES}):', CLASS_NAMES)

### Class distribution

Visualising imbalance before the split — this informs our choice of class-weighted loss.

In [ ]:
counts = labeled['failureType'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=counts.index, y=counts.values, palette='viridis', ax=ax)
ax.set_title('Class distribution — labeled wafers')
ax.set_xlabel('Failure type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
for bar, val in zip(ax.patches, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/class_distribution.png', dpi=150)
plt.show()

## 3. Train / Val / Test Split (no leakage)

The dataset ships with a `trainTestLabel` column. We use it as the primary split.  
Val is carved from the training pool using a stratified shuffle — every class is represented proportionally.

In [ ]:
train_pool = labeled[labeled['trainTestLabel'] == 'Training'].copy()
test_df    = labeled[labeled['trainTestLabel'] == 'Test'].copy()

sss = StratifiedShuffleSplit(n_splits=1, test_size=cfg.val_frac, random_state=cfg.seed)
train_idx, val_idx = next(sss.split(train_pool, train_pool['label']))

train_df = train_pool.iloc[train_idx].copy()
val_df   = train_pool.iloc[val_idx].copy()

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')

In [ ]:
# Compute mean and std from TRAINING pixels only — no test/val statistics used
train_maps = np.stack([np.asarray(w, dtype=np.float32) for w in train_df['waferMap']])
PIXEL_MEAN = float(train_maps.mean())
PIXEL_STD  = float(train_maps.std()) + 1e-8  # guard against zero std
print(f'Training pixel mean: {PIXEL_MEAN:.4f}, std: {PIXEL_STD:.4f}')

## 4. PyTorch Dataset & Transforms

Augmentations applied **only** to the training split.  
All splits share the same normalization (computed from training data).

In [ ]:
class WaferMapDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.maps   = [np.asarray(w, dtype=np.float32) for w in dataframe['waferMap']]
        self.labels = dataframe['label'].values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Shape: (H, W) → (1, H, W)
        img = torch.from_numpy(self.maps[idx]).unsqueeze(0)
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


normalize = T.Normalize(mean=[PIXEL_MEAN], std=[PIXEL_STD])

train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(degrees=(90, 90))], p=0.5),
    normalize,
])

eval_transform = T.Compose([normalize])

train_ds = WaferMapDataset(train_df, transform=train_transform)
val_ds   = WaferMapDataset(val_df,   transform=eval_transform)
test_ds  = WaferMapDataset(test_df,  transform=eval_transform)

print(f'Dataset sizes — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')

## 5. Class Weights & DataLoaders

In [ ]:
# Inverse-frequency weights — computed from training set only
label_counts = np.bincount(train_df['label'].values, minlength=N_CLASSES).astype(float)
class_weights = torch.tensor(
    len(train_df) / (N_CLASSES * label_counts), dtype=torch.float32
).to(DEVICE)

print('Class weights:')
for name, w in zip(CLASS_NAMES, class_weights.cpu().numpy()):
    print(f'  {name:<12} {w:.3f}')

In [ ]:
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)

## 6. CNN Architecture

Three conv blocks halve spatial dimensions at each stage.  
Input `(1, 45, 48)` → `(32, 22, 24)` → `(64, 11, 12)` → `(128, 5, 6)` → flatten 3,840 → FC256 → FC9.

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2),
    )

class WaferCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        # Compute flattened size dynamically — safe against rounding in MaxPool
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 45, 48)
            flat_size = self.features(dummy).numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = WaferCNN(N_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {total_params:,}')

## 7. Loss, Optimizer, Scheduler & Early Stopping

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=cfg.lr_patience, factor=cfg.lr_factor, verbose=True
)

class EarlyStopping:
    def __init__(self, patience, checkpoint_path):
        self.patience = patience
        self.path = checkpoint_path
        self.best_loss = float('inf')
        self.counter = 0
        self.stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

early_stop = EarlyStopping(cfg.es_patience, cfg.checkpoint)

## 8. Training Loop

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None):
    """One forward pass over loader. Pass optimizer=None for eval mode."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)

    return total_loss / total, correct / total


history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f'{"Epoch":>6}  {"Train Loss":>10}  {"Train Acc":>10}  {"Val Loss":>10}  {"Val Acc":>10}  {"LR":>8}')
print('-' * 65)

for epoch in range(1, cfg.max_epochs + 1):
    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer)
    vl_loss, vl_acc = run_epoch(val_loader,   model, criterion)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f'{epoch:>6}  {tr_loss:>10.4f}  {tr_acc:>10.4f}  {vl_loss:>10.4f}  {vl_acc:>10.4f}  {current_lr:>8.2e}')

    scheduler.step(vl_loss)
    early_stop(vl_loss, model)
    if early_stop.stop:
        print(f'\nEarly stopping at epoch {epoch}. Best val loss: {early_stop.best_loss:.4f}')
        break

print('Training complete.')

## 9. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_ran, history['train_loss'], label='Train')
ax1.plot(epochs_ran, history['val_loss'],   label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-entropy loss')
ax1.legend()

ax2.plot(epochs_ran, history['train_acc'], label='Train')
ax2.plot(epochs_ran, history['val_acc'],   label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/training_curves.png', dpi=150)
plt.show()

## 10. Test Set Evaluation

Load the best checkpoint saved by EarlyStopping. Test set is touched **only here**.

In [ ]:
model.load_state_dict(torch.load(cfg.checkpoint, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print('=== Test Set Results ===')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds, normalize='true')

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', values_format='.2f')
ax.set_title('Confusion matrix (normalised by true label)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/confusion_matrix.png', dpi=150)
plt.show()